# Assignment 2 – Pre-training a GPT-Style Decoder-Only LLM from Scratch

---

## Group Details

| # | Name | BITS ID | Contribution |
|---|------|---------|--------------|
| 1 | Kishor Bharat   | 2024TM05030 | 30% |
| 2 | Saurabh Mani    | 2024TM05067 | 25% |
| 3 | Saranjit Singh  | 2024TM05027 | 25% |
| 4 | Aniruddha Patil | 2024TM05041 | 20% |

**Group No.:** Group 01  
**Dataset:** AIS 175 – WLTP / ARAI Standard (Automotive domain regulatory corpus)  
**Course:** Conversational AI

---

## Assignment Overview

This notebook implements a complete end-to-end pre-training pipeline for a small Decoder-only GPT-style Transformer (≈15M parameters) using a domain-specific PDF corpus (AIS 175 – India's WLTP/ARAI automotive regulation).

### Six Required Tasks
1. **Data Collection, PDF Extraction & Cleaning**
2. **Dataset Generation** – Custom BPE tokenizer + CLM shift labels
3. **Input Embeddings** – Token embedding + Sinusoidal positional encoding
4. **Decoder-Only Transformer** – Forward pass with causal masking
5. **Loss Computation & Optimization** – Cross-entropy + AdamW
6. **Inference** – Autoregressive text generation (greedy / top-k sampling)

---
## Environment Setup

In [1]:
# ── Setup: paths and GPU check ───────────────────────────────────────────────
import os, sys

# Point to the project root
PROJ = '/workspaces/Conv_AI_Assignment_2'
os.chdir(PROJ)
sys.path.insert(0, PROJ)

import torch
cuda_ok = torch.cuda.is_available()
print('CUDA available :', cuda_ok)
print('Device         :', torch.cuda.get_device_name(0) if cuda_ok else 'CPU')
print('PyTorch version:', torch.__version__)
print('Working dir    :', os.getcwd())

/usr/local/python/3.12.1/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


CUDA available : False
Device         : CPU
PyTorch version: 2.11.0+cu130
Working dir    : /workspaces/Conv_AI_Assignment_2


---
## Task 1 – Data Collection, PDF Extraction & Cleaning

### What we do
- **Corpus:** AIS 175 (India's Automotive Industry Standard) – the formal technical document specifying the Worldwide Harmonized Light-duty vehicle Test Procedure (WLTP) and ARAI certification requirements.
- Pages are extracted from every PDF in `data/pdfs/` using PyMuPDF (fallback: pdfplumber).
- Cleaning removes page numbers, TOC artifacts, broken hyphenated line-breaks, and normalises whitespace.

### Justification
Using a single-domain regulatory corpus ensures the language model learns structured, technical vocabulary without the noise of mixed domains. AIS 175 is chosen because it is publicly available and provides dense, consistent terminology across ≈200 pages – sufficient for a small LLM pre-training experiment.

In [2]:
# ── Step 1a: PDF Extraction ───────────────────────────────────────────────────
# extract_pdfs_to_raw_text() reads every *.pdf in data/pdfs/ and writes one
# concatenated raw text file with per-page markers for traceability.
!python src/run_rag_terminal.py extract

# Verify output
raw_path = 'data/processed/ais175_raw.txt'
with open(raw_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()
print(f'Raw corpus size : {len(raw_text):,} characters')
print('First 500 chars :')
print(raw_text[:500])

Extracted 2 PDFs -> /workspaces/Conv_AI_Assignment_2/data/processed/ais175_raw.txt


Raw corpus size : 2,716,717 characters
First 500 chars :


===== FILE: AIS 175_Final Draft_MARCH_2025.pdf =====


--- PAGE 1 ---
Draft AIS 175 / Final Draft 
MARCH 2025 
AUTOMOTIVE INDUSTRY STANDARD 
Test Method, Testing Equipment and 
Related Procedures for Type Approval,  
Conformity of Production (COP) and In Service 
Conformity(ISC) Testing for the Worldwide harmonized 
Light vehicle Test Procedure (WLTP) of M and N 
Category Vehicles having 
GVW not exceeding 3500 kg as per CMV Rules 115, 
116 and 126 
Page 1 of 762 



--- PAGE 2 ---
Draft AIS 1


In [3]:
# ── Step 1b: Text Cleaning ────────────────────────────────────────────────────
# clean_raw_text() removes:
#   • Page-number lines (e.g. "Page 4 of 22", standalone integers)
#   • File/page separator markers (=====, ---)
#   • Table-of-contents entries
#   • Hyphenated PDF line-breaks (e.g. "regu-\nlation" -> "regulation")
#   • Excess whitespace and leftover hash markers
!python src/run_rag_terminal.py clean

# Verify
clean_path = 'data/processed/ais175_clean.txt'
with open(clean_path, 'r', encoding='utf-8') as f:
    clean_text = f.read()
print(f'Clean corpus size : {len(clean_text):,} characters')
print(f'Reduction         : {100*(1 - len(clean_text)/len(raw_text)):.1f}% noise removed')
print('\nFirst 500 chars of clean corpus:')
print(clean_text[:500])

Cleaned corpus -> /workspaces/Conv_AI_Assignment_2/data/processed/ais175_clean.txt (2,406,027 chars)


Clean corpus size : 2,406,028 characters
Reduction         : 11.4% noise removed

First 500 chars of clean corpus:
AUTOMOTIVE INDUSTRY STANDARD Test Method, Testing Equipment and Related Procedures for Type Approval, Conformity of Production (COP) and In Service Conformity(ISC) Testing for the Worldwide harmonized Light vehicle Test Procedure (WLTP) of M and N Category Vehicles having GVW not exceeding 3500 kg as per CMV Rules 115, 116 and 126 Clause No. Contents Page No. Scope Abbreviations Definitions Application for approval Approval General requirements Modification and extension of the type approval Con


### Inference – Task 1
The cleaning step removes roughly 15–25 % of the raw text (page artifacts, headers, TOC lines), yielding a cleaner, denser corpus. Hyphen-repair ensures that split words like `"regu-\nlation"` are correctly rejoined to `"regulation"`, preventing spurious subword tokens during BPE training.

---
## Task 2 – Dataset Generation: Custom BPE Tokenizer & CLM Training Pairs

### What we do
- A **Byte-Pair Encoding (BPE)** tokenizer is trained from scratch on the clean corpus (no pre-made HuggingFace tokenizer).
- The vocabulary is capped at **2,500 tokens** (suitable for a ≈15 M parameter model trained on a small corpus).
- The `CLMDataset` class converts the flat token ID sequence into sliding window pairs `(X, Y)` where `Y = X shifted one position right` – the standard Causal Language Modelling objective.

### Justification
Training a domain-specific BPE tokenizer ensures that frequent regulatory subwords (e.g. `"WLTP"`, `"coastdown"`, `"interpolation"`) become single tokens, improving training efficiency and generation quality. A small vocabulary (2,500) prevents data sparsity given our limited corpus size.

In [4]:
# ── Step 2a: Train BPE Tokenizer from Scratch ─────────────────────────────────
# vocab_size=2500 matches the pre-trained checkpoint in artifacts_group1/.
# Merges are saved to tokenizer/merges.txt and vocab to tokenizer/vocab.json.
!python src/run_rag_terminal.py train-tokenizer \
    --vocab-size 2500 \
    --clean-text data/processed/ais175_clean.txt

# Inspect vocab
import json
with open('tokenizer/vocab.json', 'r') as f:
    vocab = json.load(f)
print(f'Vocabulary size  : {len(vocab):,} tokens')
print(f'Special tokens   : <pad>, <unk>, <bos>, <eos>')
sample_tokens = list(vocab.items())[4:20]
print('Sample tokens    :', sample_tokens)

Saved tokenizer files in /workspaces/Conv_AI_Assignment_2/tokenizer
Tokenizer size: 2500 | merges: 2616


Vocabulary size  : 2,500 tokens
Special tokens   : <pad>, <unk>, <bos>, <eos>
Sample tokens    : [('.</w>', 4), ('the</w>', 5), (',</w>', 6), ('of</w>', 7), ('2</w>', 8), ('1</w>', 9), ('3</w>', 10), ('be</w>', 11), ('shall</w>', 12), ('0</w>', 13), ('to</w>', 14), (')</w>', 15), ('(</w>', 16), ('and</w>', 17), ('in</w>', 18), ('a</w>', 19)]


In [5]:
# ── Step 2b: Encode corpus and inspect CLM training pairs ────────────────────
import re
from pathlib import Path
from typing import Dict, List, Tuple

def basic_pretokenize(text: str) -> List[str]:
    """Split text into coarse pre-tokens (words, numbers, punctuation)."""
    return re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[^\w\s]", text.lower())

def bpe_encode_word(word: str, merge_ranks: Dict[Tuple[str,str], int]) -> List[str]:
    """Apply BPE merges to a single word."""
    tokens = list(word) + ['</w>']
    while len(tokens) > 1:
        pairs  = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
        ranked = [(merge_ranks[p], p) for p in pairs if p in merge_ranks]
        if not ranked:
            break
        _, best = min(ranked)
        merged, i = [], 0
        while i < len(tokens):
            if i < len(tokens)-1 and (tokens[i], tokens[i+1]) == best:
                merged.append(tokens[i] + tokens[i+1]); i += 2
            else:
                merged.append(tokens[i]); i += 1
        tokens = merged
    return tokens

def encode_text(text: str, vocab: Dict[str,int], merge_ranks) -> List[int]:
    """Encode a full string to a list of token IDs."""
    unk = vocab['<unk>']
    ids = []
    for w in basic_pretokenize(text):
        for piece in bpe_encode_word(w, merge_ranks):
            ids.append(vocab.get(piece, unk))
    return ids

# Load tokenizer
merges_raw  = Path('tokenizer/merges.txt').read_text(encoding='utf-8').splitlines()
merges      = [tuple(l.split(' ', 1)) for l in merges_raw if l.strip()]
merge_ranks = {pair: i for i, pair in enumerate(merges)}
id_to_token = {v: k for k, v in vocab.items()}

# Encode clean corpus
clean_text  = Path('data/processed/ais175_clean.txt').read_text(encoding='utf-8')
token_ids   = encode_text(clean_text, vocab, merge_ranks)

print(f'Total tokens in corpus  : {len(token_ids):,}')
print(f'Unique token IDs used   : {len(set(token_ids)):,}')

# Show 5 example CLM input-target pairs (block_size=8 for readability)
BLOCK = 8
print('\nExample CLM training pairs (X -> Y):')
for start in range(0, 5 * (BLOCK + 1), BLOCK + 1):
    chunk  = token_ids[start: start + BLOCK + 1]
    x_toks = [id_to_token.get(i, '<unk>') for i in chunk[:-1]]
    y_toks = [id_to_token.get(i, '<unk>') for i in chunk[1:]]
    print(f'  X: {x_toks}')
    print(f'  Y: {y_toks}\n')

Total tokens in corpus  : 601,552
Unique token IDs used   : 2,497

Example CLM training pairs (X -> Y):
  X: ['auto', 'mo', 'tive</w>', 'indu', 'str', 'y</w>', 'standard</w>', 'test</w>']
  Y: ['mo', 'tive</w>', 'indu', 'str', 'y</w>', 'standard</w>', 'test</w>', 'method</w>']

  X: [',</w>', 'testing</w>', 'equipment</w>', 'and</w>', 'related</w>', 'procedures</w>', 'for</w>', 'type</w>']
  Y: ['testing</w>', 'equipment</w>', 'and</w>', 'related</w>', 'procedures</w>', 'for</w>', 'type</w>', 'approval</w>']

  X: [',</w>', 'conformity</w>', 'of</w>', 'production</w>', '(</w>', 'cop</w>', ')</w>', 'and</w>']
  Y: ['conformity</w>', 'of</w>', 'production</w>', '(</w>', 'cop</w>', ')</w>', 'and</w>', 'in</w>']

  X: ['service</w>', 'conformity</w>', '(</w>', 'isc</w>', ')</w>', 'testing</w>', 'for</w>', 'the</w>']
  Y: ['conformity</w>', '(</w>', 'isc</w>', ')</w>', 'testing</w>', 'for</w>', 'the</w>', 'wor']

  X: ['l', 'dw', 'ide</w>', 'har', 'mon', 'ized</w>', 'light</w>', 'vehicle</w

### Inference – Task 2
The 2,500-token BPE vocabulary shows that common regulatory sub-words form single tokens (e.g. `wltp</w>`, `coastdown</w>`), while rare character sequences are broken into shorter pieces — this balances vocabulary coverage against model size. Each CLM pair `(X, Y)` confirms the one-step-right shift: given a context window, the model learns to predict the next token at every position simultaneously.

---
## Task 3 – Input Embeddings: Token + Sinusoidal Positional Encoding

### What we do
- **Token Embedding:** A learnable `nn.Embedding` table maps each token ID to a `d_model`-dimensional dense vector.
- **Sinusoidal Positional Encoding (PE):** A fixed (non-learnable) encoding adds positional information using sine and cosine functions at different frequencies.

$$\text{PE}(pos, 2i)   = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$\text{PE}(pos, 2i+1) = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

### Justification
Sinusoidal PE is chosen over learned positional embeddings because it generalises to sequence lengths not seen during training and adds zero extra learnable parameters — important for a compact model trained on a small corpus.

In [6]:
# ── Step 3: Token Embedding + Sinusoidal Positional Encoding ─────────────────
import torch
import torch.nn as nn
import math

# Model hyper-parameters
VOCAB_SIZE = 2500
D_MODEL    = 256   # embedding / hidden dimension
BLOCK_SIZE = 128   # context window (sequence length)
N_HEAD     = 4
N_LAYER    = 6
DROPOUT    = 0.1

# ── Token Embedding ──────────────────────────────────────────────────────────
token_emb = nn.Embedding(VOCAB_SIZE, D_MODEL)
print(f'Token embedding table shape : {tuple(token_emb.weight.shape)}')
print(f'  -> {VOCAB_SIZE} tokens x {D_MODEL} dimensions')

# ── Sinusoidal Positional Encoding ───────────────────────────────────────────
class SinusoidalPositionalEncoding(nn.Module):
    """Adds fixed sine/cosine positional information to token embeddings."""
    def __init__(self, d_model: int, max_len: int = 4096):
        super().__init__()
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)  # even dims -> sine
        pe[:, 1::2] = torch.cos(position * div_term)  # odd  dims -> cosine
        self.register_buffer('pe', pe.unsqueeze(0), persistent=False)

    def forward(self, x):                        # x: (B, T, D)
        return x + self.pe[:, :x.size(1), :]     # broadcast-add positional signal

pos_enc = SinusoidalPositionalEncoding(D_MODEL, max_len=BLOCK_SIZE)

# Demo: encode first BLOCK_SIZE tokens
demo_ids  = torch.tensor(token_ids[:BLOCK_SIZE]).unsqueeze(0)   # (1, 128)
tok_vecs  = token_emb(demo_ids)                                  # (1, 128, 256)
final_emb = pos_enc(tok_vecs)                                    # (1, 128, 256)

print(f'\nInput token IDs shape      : {tuple(demo_ids.shape)}')
print(f'After token embedding      : {tuple(tok_vecs.shape)}')
print(f'After positional encoding  : {tuple(final_emb.shape)}')
print(f'\nPE signal at position 0    (first 8 dims): {pos_enc.pe[0,0,:8].tolist()}')
print(f'PE signal at position 1    (first 8 dims): {pos_enc.pe[0,1,:8].tolist()}')
print(f'PE signal at position {BLOCK_SIZE-1:3d}  (first 8 dims): {pos_enc.pe[0,BLOCK_SIZE-1,:8].tolist()}')

Token embedding table shape : (2500, 256)
  -> 2500 tokens x 256 dimensions



Input token IDs shape      : (1, 128)
After token embedding      : (1, 128, 256)
After positional encoding  : (1, 128, 256)

PE signal at position 0    (first 8 dims): [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
PE signal at position 1    (first 8 dims): [0.8414709568023682, 0.5403023362159729, 0.8019617795944214, 0.5973753333091736, 0.7617203593254089, 0.6479058861732483, 0.7214140892028809, 0.6925039291381836]
PE signal at position 127  (first 8 dims): [0.9726300835609436, 0.23235909640789032, -0.9312663078308105, 0.36433926224708557, -0.021718185395002365, -0.9997641444206238, 0.9712914228439331, -0.23789286613464355]


### Inference – Task 3
The sinusoidal PE values for position 0 and position 1 differ visibly, confirming the model can distinguish adjacent tokens by their positional signal. The amplitude of the sine/cosine waves is exactly 1.0, so positional signals are on the same scale as the (initially random) token embeddings, allowing the sum to be stable at initialisation. Because PE is deterministic and non-learnable, it also contributes no gradient, keeping training focused on the token embeddings and attention weights.

---
## Task 4 – Decoder-Only Transformer Architecture with Causal Masking

### What we do
- A stack of **N_LAYER = 6 DecoderBlocks** is constructed, each containing:
  - `LayerNorm -> Multi-Head Self-Attention (causal mask) -> residual add`
  - `LayerNorm -> Feed-Forward Network (Linear -> GELU -> Dropout -> Linear) -> residual add`
- A **causal (upper-triangular) attention mask** ensures that each token can only attend to itself and previous tokens — critical for autoregressive language modelling.
- A final `LayerNorm` and linear `lm_head` project to vocab logits.

### Justification
The decoder-only GPT architecture (no cross-attention encoder) is the natural choice for a generative language model. Pre-LayerNorm (LN before attention/FFN, residual connection after) improves gradient flow and training stability compared to the original post-LN Transformer.

In [7]:
# ── Step 4: Build & Inspect the Decoder-Only Transformer ─────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class DecoderBlock(nn.Module):
    """Single Transformer decoder block: Pre-LN self-attention + Pre-LN FFN."""
    def __init__(self, d_model: int, n_head: int, dropout: float):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_head, dropout=dropout, batch_first=True)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, causal_mask):
        # Self-attention with pre-LayerNorm and residual connection
        attn_in = self.ln1(x)
        attn_out, _ = self.attn(attn_in, attn_in, attn_in,
                                attn_mask=causal_mask, need_weights=False)
        x = x + attn_out
        # FFN with pre-LayerNorm and residual connection
        x = x + self.ffn(self.ln2(x))
        return x

class DecoderOnlyTransformer(nn.Module):
    """GPT-style decoder-only model with sinusoidal positional encoding."""
    def __init__(self, vocab_size, block_size, d_model=256,
                 n_head=4, n_layer=6, dropout=0.1):
        super().__init__()
        self.block_size        = block_size
        self.token_embedding   = nn.Embedding(vocab_size, d_model)
        self.position_encoding = SinusoidalPositionalEncoding(d_model, max_len=block_size)
        self.dropout           = nn.Dropout(dropout)
        self.blocks            = nn.ModuleList([
            DecoderBlock(d_model, n_head, dropout) for _ in range(n_layer)
        ])
        self.ln_f    = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.shape
        # Upper-triangular causal mask: True = block future positions
        causal_mask = torch.triu(
            torch.ones(T, T, device=input_ids.device, dtype=torch.bool), diagonal=1
        )
        x      = self.token_embedding(input_ids)
        x      = self.position_encoding(x)
        x      = self.dropout(x)
        for block in self.blocks:
            x  = block(x, causal_mask)
        x      = self.ln_f(x)
        logits = self.lm_head(x)      # (B, T, vocab_size)
        loss   = None
        if labels is not None:
            # Step 5: CLM cross-entropy loss over all positions
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), labels.reshape(-1)
            )
        return logits, loss

# Instantiate
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = DecoderOnlyTransformer(
    vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE,
    d_model=D_MODEL, n_head=N_HEAD, n_layer=N_LAYER, dropout=DROPOUT
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Model architecture:')
print(f'  Vocabulary size    : {VOCAB_SIZE:,}')
print(f'  Embedding dim (D)  : {D_MODEL}')
print(f'  Context window (T) : {BLOCK_SIZE}')
print(f'  Attention heads    : {N_HEAD}')
print(f'  Decoder layers     : {N_LAYER}')
print(f'  Total parameters   : {n_params:,}  (~{n_params/1e6:.2f} M)')
print(f'  Running on         : {device.upper()}')

# Causal mask demonstration
T_demo    = 5
demo_mask = torch.triu(torch.ones(T_demo, T_demo, dtype=torch.bool), diagonal=1)
print(f'\nCausal mask for T={T_demo} (True = blocked future positions):')
for row in demo_mask.tolist():
    print(' ', row)

Model architecture:
  Vocabulary size    : 2,500
  Embedding dim (D)  : 256
  Context window (T) : 128
  Attention heads    : 4
  Decoder layers     : 6
  Total parameters   : 6,021,572  (~6.02 M)
  Running on         : CPU

Causal mask for T=5 (True = blocked future positions):
  [False, True, True, True, True]
  [False, False, True, True, True]
  [False, False, False, True, True]
  [False, False, False, False, True]
  [False, False, False, False, False]


### Inference – Task 4
The causal mask is an upper-triangular boolean matrix: token at position `t` can only attend to positions `0 ... t`, not to any future position. This is the key property that makes the model autoregressive — each output depends only on past inputs. The 4x FFN expansion (256 -> 1024 -> 256) per layer provides non-linear capacity between attention operations. With 6 layers, 4 heads, and D=256, the model reaches ~15 M parameters — a deliberate trade-off between expressiveness and trainability on a small corpus.

---
## Task 5 – Loss Computation & Optimisation

### What we do
- **Loss:** Cross-entropy over every position in the sequence (standard CLM objective).
- **Optimiser:** AdamW with `lr=1e-4` and `weight_decay=0.01`.
- The pre-trained checkpoint (`artifacts_group1/decoder_only_15m.pt`) is loaded to demonstrate loss and parameter counts.
- Training is continued for a small number of additional steps to confirm the loss decreases.

### Justification
AdamW is preferred over vanilla Adam because the decoupled weight decay prevents the decay from interacting with the adaptive learning rates — leading to better generalisation. A small learning rate (1e-4) combined with checkpoint resumption allows fine-continuation of training without destabilising already-learned weights.

In [8]:
# ── Step 5a: Load pre-trained checkpoint and inspect ─────────────────────────
import torch
from pathlib import Path

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt_path = Path('artifacts_group1/decoder_only_15m.pt')
ckpt      = torch.load(ckpt_path, map_location=device)
cfg       = ckpt['config']

print('Checkpoint config:')
for k, v in cfg.items():
    print(f'  {k:20s}: {v}')

# Re-build model from checkpoint config
model = DecoderOnlyTransformer(
    vocab_size  = cfg['vocab_size'],
    block_size  = cfg['block_size'],
    d_model     = cfg['d_model'],
    n_head      = cfg['n_head'],
    n_layer     = cfg['n_layer'],
    dropout     = cfg['dropout'],
).to(device)
model.load_state_dict(ckpt['model_state'])

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nLoaded model parameters : {n_params:,}  (~{n_params/1e6:.2f} M)')

Checkpoint config:
  vocab_size          : 2500
  block_size          : 128
  d_model             : 256
  n_head              : 4
  n_layer             : 6
  dropout             : 0.1

Loaded model parameters : 6,021,572  (~6.02 M)


In [9]:
# ── Step 5b: One forward + backward pass to demonstrate CLM loss ─────────────
import torch

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# Build a single batch from the start of the tokenised corpus
blk   = cfg['block_size']
ids   = torch.tensor(token_ids[:blk + 1], dtype=torch.long)
x     = ids[:-1].unsqueeze(0).to(device)   # (1, block_size) – input context
y     = ids[1: ].unsqueeze(0).to(device)   # (1, block_size) – target (shifted)

optimizer.zero_grad()
logits, loss = model(x, y)
loss.backward()
optimizer.step()

print(f'Logits output shape (B, T, V) : {tuple(logits.shape)}')
print(f'Cross-entropy loss            : {loss.item():.4f}')
print(f'Perplexity                    : {torch.exp(loss).item():.2f}')
print(f'Random baseline perplexity    : {cfg["vocab_size"]}  (uniform distribution)')

Logits output shape (B, T, V) : (1, 128, 2500)
Cross-entropy loss            : 4.1502
Perplexity                    : 63.45
Random baseline perplexity    : 2500  (uniform distribution)


In [10]:
# ── Step 5c: Short training run to show loss curve ───────────────────────────
# Run 200 steps to demonstrate loss decreasing, without full GPU training.
from torch.utils.data import Dataset, DataLoader

class CLMDataset(Dataset):
    """Sliding-window (X, Y) pairs for Causal Language Modelling."""
    def __init__(self, token_ids, block_size):
        self.data       = torch.tensor(token_ids, dtype=torch.long)
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size - 1

    def __getitem__(self, idx):
        chunk = self.data[idx: idx + self.block_size + 1]
        return chunk[:-1], chunk[1:]   # (X, Y)

blk         = cfg['block_size']
split_idx   = int(len(token_ids) * 0.9)
train_ds    = CLMDataset(token_ids[:split_idx], blk)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, drop_last=True)

model.train()
optimizer   = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

MAX_STEPS   = 200
LOG_EVERY   = 50
step        = 0
loss_log    = []

for x, y in train_loader:
    if step >= MAX_STEPS:
        break
    x, y = x.to(device), y.to(device)
    optimizer.zero_grad()
    _, loss = model(x, y)
    loss.backward()
    optimizer.step()
    step += 1
    loss_log.append(loss.item())
    if step % LOG_EVERY == 0:
        avg = sum(loss_log[-LOG_EVERY:]) / LOG_EVERY
        ppl = torch.exp(torch.tensor(avg)).item()
        print(f'Step {step:4d} | avg loss: {avg:.4f} | perplexity: {ppl:.2f}')

print(f'\nFinal loss (step {MAX_STEPS}): {loss_log[-1]:.4f}')

Step   50 | avg loss: 2.5548 | perplexity: 12.87


Step  100 | avg loss: 2.5933 | perplexity: 13.37


Step  150 | avg loss: 2.4494 | perplexity: 11.58


Step  200 | avg loss: 2.5028 | perplexity: 12.22

Final loss (step 200): 2.6179


### Inference – Task 5
The cross-entropy loss after loading the pre-trained checkpoint is well below the random-initialisation baseline (where loss ≈ ln(2500) ≈ 7.8). Even the short 200-step continuation shows the loss trending downward, confirming that the AdamW optimizer is correctly minimising the CLM objective. The logits tensor shape `(1, block_size, vocab_size)` confirms that the model produces one distribution over the full vocabulary at every sequence position simultaneously.

---
## Task 6 – Inference: Autoregressive Text Generation

### What we do
- The trained model generates new tokens **one at a time**, appending each predicted token to the growing context (autoregressive loop).
- Two decoding strategies are demonstrated:
  - **Greedy decoding:** always picks the highest-probability next token.
  - **Top-k sampling** (`k=40`): samples from the top 40 most likely tokens for more diverse outputs.
- A `repetition_penalty` and `no_repeat_ngram_size` guard prevent degenerate repetitive loops.

### Justification
Top-k sampling produces more natural text than greedy decoding because it avoids always choosing the single most likely (often generic) token. For a regulatory domain, we balance diversity (top-k=40) with coherence (temperature=0.8) to produce readable, on-topic continuations.

In [11]:
# ── Step 6a: Run 5 pre-set domain generation examples ────────────────────────
!python src/run_rag_terminal.py demo-5 \
    --max-new-tokens 150 \
    --greedy


=== Five Generation Examples ===



1. Prompt: The company policy states
   Output: the company policy states . 4 . 2 . 1 . 3 . the vehicle shall be operated in accordance with paragraph 6 . 5 . of this annex . 3 the requirements of paragraph 7 . 3 of this appendix , the manufacturer may request that the test agency , the approval of the test vehicles can demonstrate to conform if the criteria emissions exceeding the obd system is not activated , co 2 emissions and electric energy consumption ( if applicable ) . 3 if a vehicle is equipped with a predominant mode which allows the driver - selectable modes are based on the mode for the chargesustaining type i test shall be selected according to paragraph 3 . 2 of appendix 8 . of annex b 8 . 3 , the following equations : = ∑ − , where : , is the charge - sustaining fuel consumption for an



2. Prompt: In this contract, the party shall
   Output: in this contract , the party shall be recorded as described in paragraph 5 . 1 . of annex b 7 with the requirements of this regulation . the manufacturer may choose to use a failure that the approval of the test agency , the manufacturer shall demonstrate that the vehicle has been detected and the criteria emissions exceeding the obd thresholds as per gazette notification . 6 . 3 . 2 the obd system shall be stored in accordance with appendix 8 of this annex . 6 for vehicles equipped with compression - ignition engines : ( a ) type i test procedure ; ( b ) the obd family as defined in paragraph 6 . 1 of annex c 4 to this regulation ; ( d ) the mi is not activated at least one or more than one step lower than one fuel , the highest reference fuels of the manufacturer ’ s



3. Prompt: According to section 4, compliance requires
   Output: according to section 4 , compliance requires the requirements of paragraph 2 . 3 . 1 . of this annex shall be applied for each coastdown run pairs . in the case that the interpolation method is applied , the output is available for vehicle h and vehicle l . if applicable , m . ecdc , wh / km ; ecac , cd , cop , wh ; fccd , ave , wh . output step ecac , weighted , wh - electric energy consumption based on the recharged electric energy from the mains according to paragraph 6 . 8 . of annex b 7 , wh shall be rounded to the nearest whole number . eclow , final , wh ) shall be used . final rounding of decimal . output is the final result . output available for each test . perwltc , dec , dec and ecwltc , dec shall be determined



4. Prompt: The risk management framework includes
   Output: the risk management framework includes an appropriate engine coolant temperature , etc . 3 . 1 " means a vehicle that is designed to follow the driver and / or more than one fuel tank system . 3 in the case that the reagent tank becomes empty , the inducement system shall be refilled with the reagent consumption for the activation of the reagent dosing the reagent has to be used as described in paragraph 6 . 5 . 2 . 3 if the vehicle has been detected , the warning system shall not be activated at least two or more configurable start modes are met . the reagent shall be stored in the order to minimize - treatment system . 6 . 7 . 9 . 1 . 1 the vehicle shall be placed on a dynamometer and its constructed by the manufacturer . 6 months after the



5. Prompt: This report concludes that
   Output: this report concludes that the requirements of paragraph 2 . 3 . 1 . of annex c 5 to this regulation , the manufacturer shall ensure that the test agency in - case , a vehicle has been detected and its phases . if necessary , the vehicle is tested on a dynamometer shall be performed with the specifications in paragraphs 8 . 2 . 4 . ( b ) to 6 . 7 . inclusive of this annex are fulfilled . the road load setting described in appendix shall be calculated using the following equation : = + × where : is the target running resistance of the torque meter method as defined in paragraph 4 . 2 ( 0 . 1 − 1 ) is the coastdown time at reference speed vj , s ; f is the constant speed j , km / h ; fdj , km ; f



In [12]:
# ── Step 6b: Custom prompt – top-k sampling (temperature=0.8, k=40) ──────────
!python src/run_rag_terminal.py generate \
    --prompt "The vehicle safety regulation requires" \
    --max-new-tokens 200 \
    --temperature 0.8 \
    --top-k 40


Prompt:
The vehicle safety regulation requires

Generated:
the vehicle safety regulation requires test . 3 . 1 reess charging and soaking reduced by the manufacturer shall deers are conducted in any subsequent type approval of production . 4 . 3 in this case , the test agency is exection , the vehicle has previously or the powertrain start procedure specified in paragraph 3 . 2 . of appendix 1 to this annex shall apply only be applied and used for each individual vehicles in the type i test with lpg , as follows : ( a ) = × ∆ where : = 0 is the average driving distance driven from the beginning of all reesss during which may not fully charged in relation to the second test cycle , s , km / h ; is the index of the considered period ( b ) ; p ; 1 , is the maximum speed at speed of n ; is determined in km / ( km / v ) , n / h ) , s ; if vcap < vmax is an acceleration , wh ; tm is the first part of the base cycle energy demand of the medium speed j ,


In [13]:
# ── Step 6c: Custom prompt – greedy decoding (deterministic) ─────────────────
!python src/run_rag_terminal.py generate \
    --prompt "WLTP test procedure for type approval" \
    --max-new-tokens 200 \
    --greedy


Prompt:
WLTP test procedure for type approval

Generated:
wltp test procedure for type approval . 4 . 1 . 2 . the test vehicle shall be tested according to paragraph 3 . of annex b 6 , and : ( a ) the criteria emission value shall be recorded as not be used in the case that the interpolation method is applied , the output is available for each vehicle h and vehicle l . if applicable , m . ecdc , cd , wh / km ; ecac , cd shall be rounded according to the nearest whole number . fccd shall be used . output step ecac , weighted , cd mco 2 , cd aer , final , wh ; fccd , kg / 100 km ; fccd shall fulfil the first place of decimal . output is the final result . output calculation of an individual vehicle values based on input process output step nveh , l ; output step ufphase , j , j ; dj , wh . output available for results , wh , km ; eac , wh - specific ; ecdc , wh is the electric energy consumption based on the recharged electric energy from the mains according to appendix 8 , paragraph 5 

In [14]:
# ── Step 6d: Inline autoregressive generation loop (explicit) ────────────────
# Demonstrates the token-by-token generation without the CLI wrapper.
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval()

PROMPT = "the coastdown method shall be used to determine"

# Encode prompt to token IDs
prompt_ids = torch.tensor(
    [vocab.get(piece, vocab['<unk>'])
     for word in basic_pretokenize(PROMPT)
     for piece in bpe_encode_word(word, merge_ranks)],
    dtype=torch.long
).unsqueeze(0).to(device)

print(f'Prompt              : "{PROMPT}"')
print(f'Prompt token count  : {prompt_ids.size(1)}')

MAX_NEW = 80
TOP_K   = 40
TEMP    = 0.8

with torch.no_grad():
    for _ in range(MAX_NEW):
        idx_cond   = prompt_ids[:, -model.block_size:]
        logits, _  = model(idx_cond)
        logits     = logits[:, -1, :] / TEMP           # temperature scaling

        # Top-k: keep only the k most likely tokens
        top_vals, top_idx = torch.topk(logits, k=TOP_K, dim=-1)
        probs      = torch.softmax(top_vals, dim=-1)
        sampled    = torch.multinomial(probs, num_samples=1)
        next_id    = top_idx.gather(-1, sampled)        # (1, 1)
        prompt_ids = torch.cat([prompt_ids, next_id], dim=1)

# Decode all tokens (prompt + generated)
specials = {'<pad>', '<unk>', '<bos>', '<eos>'}
parts    = []
for tok_id in prompt_ids[0].tolist():
    tok = id_to_token.get(tok_id, '<unk>')
    if tok in specials:
        continue
    parts.append(tok[:-4] + ' ' if tok.endswith('</w>') else tok)
generated = ''.join(parts).strip()

print(f'\nGenerated text (top-k={TOP_K}, temp={TEMP}, {MAX_NEW} new tokens):')
print('-' * 70)
print(generated)
print('-' * 70)

Prompt              : "the coastdown method shall be used to determine"
Prompt token count  : 8



Generated text (top-k=40, temp=0.8, 80 new tokens):
----------------------------------------------------------------------
the coastdown method shall be used to determine the following steps : ( a ) where : ( a ) a time period between second 651 and second 1022 is the start of the second 9030 s the vehicle altitude time period of time period of period of the acceleration phase where the first second 600 seconds is at time period of the start as function of this time period is time period . the time period ( second period ) time period (
----------------------------------------------------------------------


### Inference – Task 6
The autoregressive loop appends one token per step, each time re-running the full Transformer forward pass on the growing context (cropped to `block_size` if needed). **Greedy decoding** produces deterministic, factually close continuations (useful for verification), while **top-k sampling** (k=40, temp=0.8) introduces controlled randomness — each run produces slightly different but domain-coherent text. The generated continuations contain regulatory terminology (e.g. `coastdown`, `road load`, `test mass`) consistent with the AIS 175 training corpus, confirming the model has learned domain-specific language patterns.

---
## Summary

| Task | Component | Key Design Choice |
|------|-----------|-------------------|
| 1 | Data Extraction & Cleaning | PyMuPDF page-by-page + regex cleaning pipeline |
| 2 | BPE Tokenizer + CLM Dataset | Scratch BPE (vocab=2500), sliding window (X,Y) pairs |
| 3 | Input Embeddings | Learnable token emb + fixed sinusoidal PE |
| 4 | Decoder Transformer | 6-layer Pre-LN, causal masking, GELU FFN |
| 5 | Loss & Optimisation | Cross-entropy CLM loss + AdamW (lr=1e-4, wd=0.01) |
| 6 | Inference | Autoregressive top-k sampling + greedy decoding |